# Load Finetuned TAPE Model & Predict on Specific Pairs

Load the best checkpoint from finetuning and run inference on arbitrary
(kinase_seq, substrate_seq) pairs.

In [88]:
import re
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from tape import ProteinBertForSequenceClassification, TAPETokenizer

## 2. Prediction function

In [89]:
WHITESPACE_RE = re.compile(r"\s+")

def sanitize_seq(s: str) -> str:
    """Remove whitespace, uppercase."""
    if s is None:
        return ""
    return WHITESPACE_RE.sub("", str(s).upper())


def encode_pair(kinase_seq: str, substrate_seq: str,
                tokenizer: TAPETokenizer, max_len: int = 1024,
                with_sep: bool = True):
    """
    Tokenize a single (kinase, substrate) pair into input_ids + mask.
    Same encoding as training: [CLS] kin_toks <sep> sub_toks [SEP]
    """
    kin = sanitize_seq(kinase_seq)
    sub = sanitize_seq(substrate_seq)

    kin_toks = tokenizer.tokenize(kin)
    sub_toks = tokenizer.tokenize(sub)
    toks = kin_toks + (["<sep>"] if with_sep else []) + sub_toks
    toks = tokenizer.add_special_tokens(toks)
    ids = tokenizer.convert_tokens_to_ids(toks)[:max_len]

    input_ids  = torch.tensor([ids], dtype=torch.long)
    input_mask = torch.ones_like(input_ids)
    return input_ids, input_mask


@torch.no_grad()
def predict_pair(kinase_seq: str, substrate_seq: str,
                 model, tokenizer, device,
                 threshold: float = 0.5,
                 max_len: int = 1024) -> dict:
    """
    Predict interaction probability for a single (kinase, substrate) pair.

    Returns:
        dict with prob, predicted_label, logits
    """
    model.eval()
    input_ids, input_mask = encode_pair(kinase_seq, substrate_seq,
                                        tokenizer, max_len=max_len)
    input_ids  = input_ids.to(device)
    input_mask = input_mask.to(device)

    # dummy target (needed by TAPE's forward signature)
    dummy_target = torch.zeros(1, dtype=torch.long, device=device)

    (_, _), logits = model(input_ids=input_ids, input_mask=input_mask,
                           targets=dummy_target)
    prob = torch.softmax(logits, dim=-1)[0, 1].item()

    return {
        "prob": prob,
        "predicted_label": int(prob >= threshold),
        "logits": logits[0].cpu().numpy(),
    }


@torch.no_grad()
def predict_batch(pairs: list, model, tokenizer, device,
                  threshold: float = 0.5, max_len: int = 1024,
                  batch_size: int = 32) -> pd.DataFrame:
    """
    Predict on a list of (kinase_seq, substrate_seq) tuples.

    Returns:
        DataFrame with columns: kinase_seq, substrate_seq, prob, predicted_label
    """
    model.eval()
    pad_id = tokenizer.vocab.get("<pad>", 0)

    all_probs = []
    all_logits = []
    for start in tqdm(range(0, len(pairs), batch_size)):
        chunk = pairs[start : start + batch_size]

        # encode all pairs in this chunk
        ids_list = []
        for kin, sub in chunk:
            kin_s = sanitize_seq(kin)
            sub_s = sanitize_seq(sub)
            kin_toks = tokenizer.tokenize(kin_s)
            sub_toks = tokenizer.tokenize(sub_s)
            toks = kin_toks + ["<sep>"] + sub_toks
            toks = tokenizer.add_special_tokens(toks)
            ids = tokenizer.convert_tokens_to_ids(toks)[:max_len]
            ids_list.append(ids)

        # pad to max length in batch
        Lmax = max(len(x) for x in ids_list)
        input_ids  = torch.full((len(ids_list), Lmax), pad_id, dtype=torch.long)
        input_mask = torch.zeros((len(ids_list), Lmax), dtype=torch.long)
        for i, ids in enumerate(ids_list):
            n = len(ids)
            input_ids[i, :n]  = torch.tensor(ids, dtype=torch.long)
            input_mask[i, :n] = 1

        input_ids  = input_ids.to(device)
        input_mask = input_mask.to(device)
        dummy_target = torch.zeros(len(ids_list), dtype=torch.long, device=device)

        (_, _), logits = model(input_ids=input_ids, input_mask=input_mask,
                               targets=dummy_target)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())
        all_logits.extend(logits.tolist())

    df = pd.DataFrame({
        "kinase_seq":      [p[0] for p in pairs],
        "substrate_seq":   [p[1] for p in pairs],
        "prob":            all_probs,
        "logit":            all_logits,
        "predicted_label": [int(p >= threshold) for p in all_probs],
    })
    return df

## 1. Load checkpoint & rebuild model

In [90]:
CKPT_PATH = "/ksi_ml/runs/finetune_tape_v2_phosphopca_exp6/checkpoints/best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# load checkpoint
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print(f"Checkpoint keys: {list(ckpt.keys())}")
print(f"Trained for {ckpt.get('epoch', '?')} epochs")
print(f"Best val score: {ckpt.get('best_val', '?')}")
print(f"Saved threshold: {ckpt.get('threshold', 0.5)}")

# rebuild model architecture
cfg = ckpt.get("cfg", {})
tape_model_name = cfg.get("tape_model_name", "bert-base")

model = ProteinBertForSequenceClassification.from_pretrained(
    tape_model_name, num_labels=2
)

# load trained weights
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
if missing:
    print(f"Missing keys: {missing}")
if unexpected:
    print(f"Unexpected keys: {unexpected}")
else:
    print("All weights loaded successfully.")

model = model.to(device)
model.eval()

# tokenizer
tokenizer = TAPETokenizer(vocab="iupac")

# saved threshold from validation
THRESHOLD = ckpt.get("threshold", 0.5)
print(f"Using threshold: {THRESHOLD:.4f}")

Device: cuda
Checkpoint keys: ['cfg', 'state_dict', 'epoch', 'best_val', 'threshold']
Trained for 3 epochs
Best val score: 0.8095783781088555
Saved threshold: 0.33204782009124756
All weights loaded successfully.
Using threshold: 0.3320


## 3. Predict on a single pair

In [91]:
# # ── EDIT THESE with your sequences ──
# kinase_seq    = "WLLSDILGQGATANVFRGRHKKTGDLFAIKVFNNISFLRPVDVQMREFEVLKKLNHKNIVKLFAIEEETTTRHKVLIMEFCPCGSLYTVLEEPSNAYGLPESEFLIVLRDVVGGMNHLRENGIVHRDIKPGNIMRVIGEDGQSVYKLTDFGAARELEDDEQFVSLYGTEEYLHPDMYERAVLRKDHQKKYGATVDLWSIGVTFYHAATGSLPFRPFEGPRRNKEVMYKIITGKPSGAISGVQKAENGPIDWSGDMPVSCSLSRGLQVLLTPVLANILEADQEKCWGFDQFFAETSDILHRMV"
# substrate_seq = "QGPSQTSTPTDVTAI"

# result = predict_pair(kinase_seq, substrate_seq, model, tokenizer, device,
#                       threshold=THRESHOLD)

# print(f"Probability of interaction: {result['prob']:.4f}")
# print(f"Predicted label (thr={THRESHOLD:.3f}): {result['predicted_label']}")
# print(f"Raw logits: {result['logits']}")

## 4. Predict on multiple pairs (batch)

In [92]:
# # ── Option A: define pairs manually ──
# pairs = [
#     ("WLLSDILGQGATANVFRGRHKKTGDLFAIKVFNNISFLRPVDVQMREFEVLKKLNHKNIVKLFAIEEETTTRHKVLIMEFCPCGSLYTVLEEPSNAYGLPESEFLIVLRDVVGGMNHLRENGIVHRDIKPGNIMRVIGEDGQSVYKLTDFGAARELEDDEQFVSLYGTEEYLHPDMYERAVLRKDHQKKYGATVDLWSIGVTFYHAATGSLPFRPFEGPRRNKEVMYKIITGKPSGAISGVQKAENGPIDWSGDMPVSCSLSRGLQVLLTPVLANILEADQEKCWGFDQFFAETSDILHRMV",
#      "QGPSQTSTPTDVTAI"),
#     ('WEMKERLGTGGFGNVIRWHNQETGEQIAIKQCRQELSPRNRERWCLEIQIMRRLTHPNVVAARDVPEGMQNLAPNDLPLLAMEYCQGGDLRKYLNQFENCCGLREGAILTLLSDIASALRYLHENRIIHRDLKPENIVLQQGEQRLIHKIIDLGYAKELDQGSLCTSFVGTLQYLAPELLEQQKYTVTVDYWSFGTLAFECITGFRPFLPNWQPVQWHSKVRQKSEVDIVVSEDLNGTVKFSSSLPYPNNLNSVLAERLEKWLQLMLMWHPRQRGTDPTYGPNGCF',
#   'PHEEPSNSLMAIECR'),
#     # add more pairs here ...
# ]

# # ── Option B: load from CSV ──
# # df_input = pd.read_csv("my_pairs.csv")  # must have kinase_seq, substrate_seq columns
# # pairs = list(zip(df_input["kinase_seq"], df_input["substrate_seq"]))

# df_results = predict_batch(pairs, model, tokenizer, device,
#                            threshold=THRESHOLD, batch_size=32)
# df_results

## 5. Predict from a CSV file

In [93]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/splits_public/train.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_v2_train.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_seq"
sub_col =  "substrate_seq"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values
df_out["predicted_label"] = df_pred["predicted_label"].values

# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

Loaded 402935 pairs from /ksi_ml/data/splits_public/train.csv
Columns: ['Unnamed: 0', 'kinase_seq', 'substrate_seq', 'label']


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 12592/12592 [30:35<00:00,  6.86it/s]


Accuracy vs ground truth: 0.8906
Saved 402935 predictions to /ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_v2_train.csv


,Unnamed: 0,kinase_seq,substrate_seq,label,prob,logit,predicted_label
0,0,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,MKKKDEGSYDLGKKP,1,0.878617,2.120876,1
1,1,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,FPLRKTASEPNLKVR,1,0.774707,1.407509,1
2,2,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,QAGSHSNSFRLSNGR,1,0.983456,4.128230,1
3,3,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,GRGGRGGSRARNLPL,1,0.859720,1.959012,1
4,4,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,DDGYMPMSPGVAPVP,1,0.930850,2.713964,1
5,5,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,GYLRKPKSMHKRFFV,1,0.774518,1.414809,1
6,6,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,SFHFKSGSLENVPNV,1,0.909869,2.425686,1
7,7,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,LKRNRYLSFHFKSGS,1,0.788592,1.476698,1
8,8,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,VGVGRAGSVRMRDLR,1,0.926382,2.631385,1
9,9,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,KKKFRTPSFLKKSKK,1,0.872142,2.051085,1


In [94]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/splits_public/val.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_v2_val.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_seq"
sub_col =  "substrate_seq"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values
df_out["predicted_label"] = df_pred["predicted_label"].values

# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

Loaded 50853 pairs from /ksi_ml/data/splits_public/val.csv
Columns: ['Unnamed: 0', 'kinase_seq', 'substrate_seq', 'label']


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 1590/1590 [04:03<00:00,  6.53it/s]


Accuracy vs ground truth: 0.8394
Saved 50853 predictions to /ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_v2_val.csv


,Unnamed: 0,kinase_seq,substrate_seq,label,prob,logit,predicted_label
0,0,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,TESITATSPASMVGG,1,0.722592,1.169226,1
1,1,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,DLRNPHPSSAFLNLI,1,0.076226,-2.160445,0
2,2,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,RYVPPSSTDRSPYEK,1,0.929818,2.698420,1
3,3,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,PAPIRRRSSNYRAYA,1,0.922863,2.574227,1
4,4,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,LMTTPSLTPFTPSLV,1,0.559825,0.490847,1
5,5,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,AWNELKGTSKESAMR,1,0.443520,0.053839,1
6,6,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,KTQTPPVSPAPQPTE,1,0.514408,0.338258,1
7,7,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,ARIIGEKSFRRSVVG,1,0.906270,2.389449,1
8,8,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,AKKKSKISASRKLQL,1,0.928111,2.651798,1
9,9,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,PPVRRRSSANYRAYA,1,0.932583,2.705771,1


In [79]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/splits_public/test.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_v2_test.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_seq"
sub_col =  "substrate_seq"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values
df_out["predicted_label"] = df_pred["predicted_label"].values

# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

Loaded 49415 pairs from /ksi_ml/data/splits_public/test.csv
Columns: ['Unnamed: 0', 'kinase_seq', 'substrate_seq', 'label']


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 1545/1545 [03:56<00:00,  6.54it/s]


Accuracy vs ground truth: 0.8391
Saved 49415 predictions to /ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_v2_test.csv


,Unnamed: 0,kinase_seq,substrate_seq,label,prob,logit,predicted_label
0,0,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,PLLARSPSTNRKYPP,1,0.850026,1.882312,1
1,1,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,RRDYDDMSPRRGPPP,1,0.826402,1.731903,1
2,2,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,QRSRKRLSQDAYRRN,1,0.936697,2.778984,1
3,3,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,ENLKEKESCASRLCA,1,0.042436,-2.776758,0
4,4,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,APIRRRSSNYRAYAT,1,0.935243,2.738691,1
5,5,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,LEPEALHTPTLMTTP,1,0.204983,-1.015268,0
6,6,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,KENIFGESRASTFCG,1,0.165163,-1.293973,0
7,7,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,RVPTANVSVVDLTCR,1,0.985949,4.296932,1
8,8,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,SPCDFRSSFRSVTPD,1,0.973804,3.679886,1
9,9,FIFHKVLGKGSFGKVLLGELKGRGEYFAIKALKKDVVLIDDDVECT...,RHIVRKRTLRRLLQE,1,0.653692,0.846268,1


In [60]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/pairs_aug_ksi_full.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_ksi_full.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_domain_seq"
sub_col =  "substrate_seq_15aa"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values
df_out["predicted_label"] = df_pred["predicted_label"].values

# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

/tmp/ipykernel_12927/2618948857.py:5: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df_input = pd.read_csv(INPUT_CSV)


Loaded 37446 pairs from /ksi_ml/data/pairs_aug_ksi_full.csv
Columns: ['Unnamed: 0', 'substrate_seq', 'kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily', 'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord', 'kinase_human_name', 'p_ind', 'p_aa', 'label', 'kinase_domain_seq', 'substrate_seq_15aa', 'p_aa_match', 'substrate']


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 1171/1171 [02:59<00:00,  6.54it/s]


Accuracy vs ground truth: 0.5990
Saved 37446 predictions to /ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_ksi_full.csv


,Unnamed: 0,substrate_seq,kinase_gene,kinase_name,pk_group,pk_family,pk_subfamily,kinase,kinase_seq,kinseq_len,...,p_ind,p_aa,label,kinase_domain_seq,substrate_seq_15aa,p_aa_match,substrate,prob,logit,predicted_label
0,0,AAAAVEEYSCEFGSA,FES,Tyrosine-protein kinase Fes/Fps,TK,Fer,NaN,P07332,MGFSSELCSPQGHGVLQQMQEAELRLLEGMRKWMAQRVKSDREYAG...,822.0,...,8,Y,1,LVLGEQIGRGNFGEVFSGRLRADNTLVAVKSCRETLPPDLKAKFLQ...,AAAAVEEYSCEFGSA,True,NaN,0.179815,-1.173910,0
1,1,AAAAVEEYSCEFGSA,FRK,Tyrosine-protein kinase FRK,TK,Src,NaN,P42685,MSNICQRLWEYLEPYLPCLSTEADKSTVIENPGALCSPQSQRHGHY...,505.0,...,8,Y,1,IQLLKRLGSGQFGEVWEGLWNNTTPVAVKTLKPGSMDPNDFLREAQ...,AAAAVEEYSCEFGSA,True,NaN,0.005076,-4.952744,0
2,2,AAAAVEEYSCEFGSA,FYN,Tyrosine-protein kinase Fyn,TK,Src,NaN,P06241,MGCVQCKDKEATKLTEERDGSLNQSSGYRYGTDPTPQHYPSFGVTS...,537.0,...,8,Y,1,LQLIKRLGNGQFGEVWMGTWNGNTKVAIKTLKPGTMSPESFLEEAQ...,AAAAVEEYSCEFGSA,True,NaN,0.089506,-1.948753,0
3,3,AAAAVEEYSCEFGSA,JAK2,Tyrosine-protein kinase JAK2,TK,Jak,NaN,O60674,MGMACLTMTEMEGTSTSSIYQNGDISGNANSMKQIDPVLQVYLYHS...,1132.0,...,8,Y,1,LKFLQQLGKGNFGSVEMCRYDPLQDNTGEVVAVKKLQHSTEEHLRD...,AAAAVEEYSCEFGSA,True,NaN,0.004778,-5.007297,0
4,4,AAAAVEEYSCEFGSA,SYK,Tyrosine-protein kinase SYK,TK,Syk,NaN,P43405,MASSGMADSANHLPFFFGNITREEAEDYLVQGGMSDGLYLLRQSRN...,635.0,...,8,Y,1,TLEDKELGSGNFGTVKKGYYQMKKVVKTVAVKILKNEANDPALKDE...,AAAAVEEYSCEFGSA,True,NaN,0.023577,-3.354477,0
5,5,AAAAVEEYSCEFGSA,SuRTK106,Tyrosine-protein kinase STYK1,TK,TK-Unique,NaN,Q6J9G0,MGMTRMLLECSLSDKLCVIQEKQYEVIIVPTLLVTIFLILLGVILW...,422.0,...,8,Y,1,SEVLEQICSGSCGPIFRANMNTGDPSKPKSVILKALKEPAGLHEVQ...,AAAAVEEYSCEFGSA,True,NaN,0.003343,-5.400781,0
6,6,AAAAVEEYSCEFGSA,TNK1,Non-receptor tyrosine-protein kinase TNK1,TK,Ack,NaN,Q13470,MLPEAGSLWLLKLLRDIQLAQFYWPILEELNVTRPEHFDFVKPEDL...,666.0,...,8,Y,1,VCRGELLGSGCFGVVHRGLWTLPSGKSVPVAVKSLRVGPEGPMGTE...,AAAAVEEYSCEFGSA,True,NaN,0.003593,-5.331919,0
7,7,AADRDEVYTIPNSLK,FES,Tyrosine-protein kinase Fes/Fps,TK,Fer,NaN,P07332,MGFSSELCSPQGHGVLQQMQEAELRLLEGMRKWMAQRVKSDREYAG...,822.0,...,8,Y,1,LVLGEQIGRGNFGEVFSGRLRADNTLVAVKSCRETLPPDLKAKFLQ...,AADRDEVYTIPNSLK,True,NaN,0.947255,2.976968,1
8,8,AADRDEVYTIPNSLK,FRK,Tyrosine-protein kinase FRK,TK,Src,NaN,P42685,MSNICQRLWEYLEPYLPCLSTEADKSTVIENPGALCSPQSQRHGHY...,505.0,...,8,Y,1,IQLLKRLGSGQFGEVWEGLWNNTTPVAVKTLKPGSMDPNDFLREAQ...,AADRDEVYTIPNSLK,True,NaN,0.908514,2.421528,1
9,9,AADRDEVYTIPNSLK,FYN,Tyrosine-protein kinase Fyn,TK,Src,NaN,P06241,MGCVQCKDKEATKLTEERDGSLNQSSGYRYGTDPTPQHYPSFGVTS...,537.0,...,8,Y,1,LQLIKRLGNGQFGEVWMGTWNGNTKVAIKTLKPGTMSPESFLEEAQ...,AADRDEVYTIPNSLK,True,NaN,0.917193,2.527130,1


In [69]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/pairs_aug_phosphopca_finetuning.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_phosphopca_finetuning.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_domain_seq"
sub_col =  "substrate_seq_15aa"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["predicted_label"] = df_pred["predicted_label"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values

# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

Loaded 12066 pairs from /ksi_ml/data/pairs_aug_phosphopca_finetuning.csv
Columns: ['Unnamed: 0', 'substrate_seq', 'binding_max', 'fp_sub', 'true_positive', 'label', 'kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily', 'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord', 'kinase_human_name', 'p_ind', 'p_aa', 'kinase_domain_seq', 'substrate_seq_15aa', 'p_aa_match']


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 378/378 [00:51<00:00,  7.29it/s]


Accuracy vs ground truth: 0.8604
Saved 12066 predictions to /ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6__pairs_aug_phosphopca_finetuning.csv


,Unnamed: 0,substrate_seq,binding_max,fp_sub,true_positive,label,kinase_gene,kinase_name,pk_group,pk_family,...,kinase_coord,kinase_human_name,p_ind,p_aa,kinase_domain_seq,substrate_seq_15aa,p_aa_match,prob,predicted_label,logit
0,0,AAQRCTISYRAPELF,0.302741,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AAQRCTISYRAPELF,True,0.240262,0,-0.828968
1,1,ADDILETSADSLEHV,0.199031,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ADDILETSADSLEHV,True,0.770446,1,1.423526
2,2,AIKIIDKTQLNPTSL,0.430039,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,T,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AIKIIDKTQLNPTSL,True,0.796080,1,1.547331
3,3,ANRTPPKSPGDPSKD,0.463911,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ANRTPPKSPGDPSKD,True,0.886032,1,2.197319
4,4,AQGPLLPSPARGLKF,0.409252,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AQGPLLPSPARGLKF,True,0.805645,1,1.625668
5,5,AQKDLSPSLPGGILQ,0.275906,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AQKDLSPSLPGGILQ,True,0.935621,1,2.786433
6,6,ARDIKNDSNYVVKGN,0.752600,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ARDIKNDSNYVVKGN,True,0.952694,1,3.105279
7,7,ARFAVAYSPARDAGN,3.026385,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ARFAVAYSPARDAGN,True,0.058106,0,-2.413442
8,8,ASCFRRLTECFLSPS,0.272293,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,T,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ASCFRRLTECFLSPS,True,0.358085,1,-0.275795
9,9,ASRFGLGSPLPSPRA,0.205736,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ASRFGLGSPLPSPRA,True,0.785039,1,1.508588


In [70]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/pairs_aug_yeast.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6___pairs_aug_yeast.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_domain_seq"
sub_col =  "substrate_seq_15aa"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["predicted_label"] = df_pred["predicted_label"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values
# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

Loaded 7063 pairs from /ksi_ml/data/pairs_aug_yeast.csv
Columns: ['Unnamed: 0', 'kinase_gene', 'substrate', 'substrate_seq', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily', 'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord', 'p_ind', 'p_aa', 'label', 'kinase_domain_seq', 'substrate_seq_15aa', 'p_aa_match']


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 221/221 [00:28<00:00,  7.68it/s]


Accuracy vs ground truth: 0.5946
Saved 7063 predictions to /ksi_ml/code/evaluation/eval_outputs/finetune_tape_v2_phosphopca_exp6___pairs_aug_yeast.csv


,Unnamed: 0,kinase_gene,substrate,substrate_seq,kinase_name,pk_group,pk_family,pk_subfamily,kinase,kinase_seq,...,kinase_coord,p_ind,p_aa,label,kinase_domain_seq,substrate_seq_15aa,p_aa_match,prob,predicted_label,logit
0,0,ABL2,AAT2,DSSYNHEYLGITGLP,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,DSSYNHEYLGITGLP,True,0.021119,0,-3.463062
1,1,ABL2,ABD1,AFRKVKQYIEPESVK,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,AFRKVKQYIEPESVK,True,0.204151,0,-1.038680
2,2,ABL2,ABF1,MDKLVVNYYEYKHPI,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,MDKLVVNYYEYKHPI,True,0.047351,0,-2.650109
3,3,ABL2,ABF2,ISERKKLYSEYQKAK,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,ISERKKLYSEYQKAK,True,0.187931,0,-1.115911
4,4,ABL2,ACO1,IDLNKEVYDFLASAT,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,IDLNKEVYDFLASAT,True,0.064299,0,-2.328118
5,5,ABL2,ACO1,YDAGENTYQAPPADR,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,YDAGENTYQAPPADR,True,0.945713,1,2.973733
6,6,ABL2,ACO2,YDLDGTEYDIPGLMM,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,YDLDGTEYDIPGLMM,True,0.119020,0,-1.651476
7,7,ABL2,ACS2,IKEHKVVYEAHNVKA,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,IKEHKVVYEAHNVKA,True,0.054094,0,-2.501606
8,8,ABL2,ACT1,GMGQKDSYVGDEAQS,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,GMGQKDSYVGDEAQS,True,0.712699,1,1.123364
9,9,ABL2,ACT1,VTHVVPIYAGFSLPH,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,VTHVVPIYAGFSLPH,True,0.021447,0,-3.465407


## Output comparison dataframes for original exp_tape_bertbase_sep_v2_long

In [71]:
CKPT_PATH = "/ksi_ml/runs/exp_tape_bertbase_sep_v2_long/checkpoints/best.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# load checkpoint
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print(f"Checkpoint keys: {list(ckpt.keys())}")
print(f"Trained for {ckpt.get('epoch', '?')} epochs")
print(f"Best val score: {ckpt.get('best_val', '?')}")
print(f"Saved threshold: {ckpt.get('threshold', 0.5)}")

# rebuild model architecture
cfg = ckpt.get("cfg", {})
tape_model_name = cfg.get("tape_model_name", "bert-base")

model = ProteinBertForSequenceClassification.from_pretrained(
    tape_model_name, num_labels=2
)

# load trained weights
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
if missing:
    print(f"Missing keys: {missing}")
if unexpected:
    print(f"Unexpected keys: {unexpected}")
else:
    print("All weights loaded successfully.")

model = model.to(device)
model.eval()

# tokenizer
tokenizer = TAPETokenizer(vocab="iupac")

# saved threshold from validation
THRESHOLD = ckpt.get("threshold", 0.5)
print(f"Using threshold: {THRESHOLD:.4f}")

Device: cuda
Checkpoint keys: ['cfg', 'state_dict', 'epoch', 'best_val', 'threshold']
Trained for 17 epochs
Best val score: 0.2373668312140114
Saved threshold: 0.7734375


/opt/conda/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


All weights loaded successfully.
Using threshold: 0.7734


In [72]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/pairs_aug_phosphopca_finetuning.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2_long__pairs_aug_phosphopca_finetuning.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_domain_seq"
sub_col =  "substrate_seq_15aa"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["predicted_label"] = df_pred["predicted_label"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values

# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

Loaded 12066 pairs from /ksi_ml/data/pairs_aug_phosphopca_finetuning.csv
Columns: ['Unnamed: 0', 'substrate_seq', 'binding_max', 'fp_sub', 'true_positive', 'label', 'kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily', 'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord', 'kinase_human_name', 'p_ind', 'p_aa', 'kinase_domain_seq', 'substrate_seq_15aa', 'p_aa_match']


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 378/378 [01:06<00:00,  5.69it/s]


Accuracy vs ground truth: 0.5383
Saved 12066 predictions to /ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2_long__pairs_aug_phosphopca_finetuning.csv


,Unnamed: 0,substrate_seq,binding_max,fp_sub,true_positive,label,kinase_gene,kinase_name,pk_group,pk_family,...,kinase_coord,kinase_human_name,p_ind,p_aa,kinase_domain_seq,substrate_seq_15aa,p_aa_match,prob,predicted_label,logit
0,0,AAQRCTISYRAPELF,0.302741,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AAQRCTISYRAPELF,True,0.994855,1,5.088504
1,1,ADDILETSADSLEHV,0.199031,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ADDILETSADSLEHV,True,0.262299,0,-1.126701
2,2,AIKIIDKTQLNPTSL,0.430039,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,T,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AIKIIDKTQLNPTSL,True,0.230632,0,-1.282511
3,3,ANRTPPKSPGDPSKD,0.463911,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ANRTPPKSPGDPSKD,True,0.062895,0,-2.629166
4,4,AQGPLLPSPARGLKF,0.409252,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AQGPLLPSPARGLKF,True,0.003920,0,-5.458543
5,5,AQKDLSPSLPGGILQ,0.275906,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,AQKDLSPSLPGGILQ,True,0.958660,1,2.931511
6,6,ARDIKNDSNYVVKGN,0.752600,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ARDIKNDSNYVVKGN,True,0.977891,1,3.588847
7,7,ARFAVAYSPARDAGN,3.026385,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ARFAVAYSPARDAGN,True,0.003127,0,-5.690886
8,8,ASCFRRLTECFLSPS,0.272293,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,T,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ASCFRRLTECFLSPS,True,0.099303,0,-2.243601
9,9,ASRFGLGSPLPSPRA,0.205736,False,True,1,MAST1,Microtubule-associated serine/threonine-protei...,AGC,MAST,...,"[['374', '647']]",MAST1_HUMAN,8,S,FDTIKLISNGAYGAVYLVRHRDTRQRFAMKKINKQNLILRNQIQQA...,ASRFGLGSPLPSPRA,True,0.034600,0,-3.283961


In [73]:
# Load a CSV with kinase_seq and substrate_seq columns
INPUT_CSV  = "/ksi_ml/data/pairs_aug_yeast.csv"     # <-- edit this
OUTPUT_CSV = "/ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2_long__pairs_aug_yeast.csv"     # <-- edit this

df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} pairs from {INPUT_CSV}")
print(f"Columns: {list(df_input.columns)}")

# figure out column names (handle both naming conventions)
kin_col =  "kinase_domain_seq"
sub_col =  "substrate_seq_15aa"

pairs = list(zip(df_input[kin_col], df_input[sub_col]))

df_pred = predict_batch(pairs, model, tokenizer, device,
                        threshold=THRESHOLD, batch_size=32)

# merge predictions back with original data
df_out = df_input.copy()
df_out["prob"]            = df_pred["prob"].values
df_out["predicted_label"] = df_pred["predicted_label"].values
df_out["logit"]            = df_pred["logit"].apply(lambda s: s[1]).values

# if ground truth labels exist, show accuracy
if "label" in df_out.columns:
    acc = (df_out["predicted_label"] == df_out["label"]).mean()
    print(f"Accuracy vs ground truth: {acc:.4f}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_out)} predictions to {OUTPUT_CSV}")
df_out.head(10)

Loaded 7063 pairs from /ksi_ml/data/pairs_aug_yeast.csv
Columns: ['Unnamed: 0', 'kinase_gene', 'substrate', 'substrate_seq', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily', 'kinase', 'kinase_seq', 'kinseq_len', 'kinase_coord', 'p_ind', 'p_aa', 'label', 'kinase_domain_seq', 'substrate_seq_15aa', 'p_aa_match']


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 221/221 [00:44<00:00,  4.98it/s]


Accuracy vs ground truth: 0.6136
Saved 7063 predictions to /ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2_long__pairs_aug_yeast.csv


,Unnamed: 0,kinase_gene,substrate,substrate_seq,kinase_name,pk_group,pk_family,pk_subfamily,kinase,kinase_seq,...,kinase_coord,p_ind,p_aa,label,kinase_domain_seq,substrate_seq_15aa,p_aa_match,prob,predicted_label,logit
0,0,ABL2,AAT2,DSSYNHEYLGITGLP,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,DSSYNHEYLGITGLP,True,0.002541,0,-5.871960
1,1,ABL2,ABD1,AFRKVKQYIEPESVK,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,AFRKVKQYIEPESVK,True,0.494549,0,-0.124019
2,2,ABL2,ABF1,MDKLVVNYYEYKHPI,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,MDKLVVNYYEYKHPI,True,0.023636,0,-3.676342
3,3,ABL2,ABF2,ISERKKLYSEYQKAK,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,ISERKKLYSEYQKAK,True,0.007113,0,-4.867393
4,4,ABL2,ACO1,IDLNKEVYDFLASAT,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,IDLNKEVYDFLASAT,True,0.135070,0,-1.894392
5,5,ABL2,ACO1,YDAGENTYQAPPADR,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,YDAGENTYQAPPADR,True,0.989551,1,4.374589
6,6,ABL2,ACO2,YDLDGTEYDIPGLMM,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,YDLDGTEYDIPGLMM,True,0.013614,0,-4.231056
7,7,ABL2,ACS2,IKEHKVVYEAHNVKA,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,IKEHKVVYEAHNVKA,True,0.003907,0,-5.471195
8,8,ABL2,ACT1,GMGQKDSYVGDEAQS,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,GMGQKDSYVGDEAQS,True,0.980412,1,3.698964
9,9,ABL2,ACT1,VTHVVPIYAGFSLPH,Abelson tyrosine-protein kinase 2,TK,Abl,NaN,P42684,MGQQVGRVGEAPGLQQPQPRGIRGSSAARPSGRRRDPAGRTTETGF...,...,"[['288', '539']]",8,Y,1,ITMKHKLGGGQYGEVYVGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,VTHVVPIYAGFSLPH,True,0.084673,0,-2.386014


## 6. Inspect model info

In [22]:
# what config was used for finetuning?
if "cfg" in ckpt:
    print("Finetuning config:")
    for k, v in sorted(ckpt["cfg"].items()):
        print(f"  {k}: {v}")

print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

Finetuning config:
  adamw_betas: (0.9, 0.999)
  adamw_eps: 1e-08
  allow_tf32: True
  backbone_lr_mult: 1.0
  batch_size: 32
  bf16: False
  distill_alpha: 0.1
  distill_temperature: 2.0
  dropout: 0.1
  early_stop_metric: auprc
  epochs: 40
  grad_accum: 1
  head_lr_mult: 1.0
  ks: (50, 100, 200)
  lr: 0.001
  max_len: 1024
  num_workers: 4
  optimizer_type: sgd
  out_dir: /ksi_ml/runs
  patience: 5
  plateau_cooldown: 0
  plateau_factor: 0.5
  plateau_min_lr: 1e-06
  plateau_mode: max
  plateau_monitor: val_auprc
  plateau_patience: 2
  plateau_threshold: 0.0
  plateau_threshold_mode: rel
  pretrained_ckpt: /ksi_ml/runs/exp_tape_bertbase_sep_v2_long/checkpoints/best.pt
  replay_csv: /ksi_ml/data/splits_public/train.csv
  replay_epoch_length: experimental
  replay_ratio: 0.2
  run_name: finetune_tape_v2_phosphopca_exp4
  scheduler_type: plateau
  sgd_momentum: 0.9
  sgd_nesterov: True
  tape_model_name: bert-base
  test2_csv: /ksi_ml/data/splits_public/test.csv
  test_csv: /ksi_ml/da